# Replicate baseline model -  CNN-based deep canonical correlation analysis autoencoder (CNN-DCCAE) 

In [ ]:
# import libraries
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn import metrics
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from braindecode.datautil.serialization import load_concat_dataset
from braindecode.datasets import BaseConcatDataset, BaseDataset
from braindecode.preprocessing import create_fixed_length_windows
from collections import Counter
from tqdm import tqdm
import plotly.express as px
import pandas as pd

In [ ]:
# Load the rescaled data
OUT_PATH = r'D:\Ana_Maria\cleaned_TUH_scaled'
tuh_preproc = load_concat_dataset(OUT_PATH, preload=False)


In [ ]:
# Count how many segment we have in each class
t = []
for ds in tuh_preproc.datasets:
    ds.target_name = int(ds.description['target'])
    ds.target_name = 'target'
    t.append(int(ds.description['target']))

Counter(t)

#epileptic: 1361, non-epileptic: 258

In [ ]:
# Train, Validation and Test subject-wise split (Make sure that a subject only appears in one split)

def get_subject_id(ds):
    """ Get the subject id from dict or from the path"""
    subj = ds.description.get("subject", None)

    # Get id from dict with his_id
    if isinstance(subj, dict) and "his_id" in subj:
        return subj["his_id"]

    # Get id from path
    path = ds.description.get("path", "")
    if isinstance(path, str):
        parts = path.split("/")
        return parts[2]

subjects = [get_subject_id(ds) for ds in tuh_preproc.datasets]
unique_subjects = sorted(set(subjects))

# Train(60%), Val(20%) and test(20%) split 
train_subj, temp_subj = train_test_split(unique_subjects, test_size=0.4, random_state=36)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.5, random_state=36)

train_subj, val_subj, test_subj = set(train_subj), set(val_subj), set(test_subj)

# Create Datasets
train_set = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in train_subj])
val_set   = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in val_subj])
test_set  = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in test_subj])


In [ ]:
# Generate compute windows
sfreq = 256 # Hz
window_size_samples = int(5 * sfreq) # 5 sec windows
window_stride_samples = int(2.5 * sfreq) # 50% overlap

def create_windows(dataset):
    """ It creates a windowed dataset from the input dataset"""
    return create_fixed_length_windows(
        dataset,
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=True,
        n_jobs=1,
    )

In [ ]:
# Create a Dataset with separated eeg and ecg signals 

class PairedEEGECGDataset(Dataset):
    """ 
    Contains the windowed dataset with the separated and nomalized EEG and ECG signals of each window 
    """
    def __init__(self, windows_dataset, eeg_indices, ecg_indices):

        self.dataset = windows_dataset
        self.eeg_idx = eeg_indices
        self.ecg_idx = ecg_indices

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        x, y, meta = self.dataset[idx]

        # convert to tensor
        x = torch.as_tensor(x, dtype=torch.float32)

        eeg = x[self.eeg_idx]
        ecg = x[self.ecg_idx]
        
        # Normalise
        eeg = (eeg - eeg.mean(dim=-1, keepdim=True)) / (eeg.std(dim=-1, keepdim=True) + 1e-8)
        ecg = (ecg - ecg.mean(dim=-1, keepdim=True)) / (ecg.std(dim=-1, keepdim=True) + 1e-8)
        
        return eeg, ecg, y, meta

In [ ]:
# channels
ecg_ch = ['EKG']
eeg_ch = [ 'C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'O1', 'O2', 'P3', 'P4', 'PZ', 'T3', 'T4', 'T5', 'T6']

# Window all 20 channels at once
train_windows = create_windows(train_set)
val_windows   = create_windows(val_set)
test_windows  = create_windows(test_set)
print('Windows created')

# Get channel indices
ch_names    = train_windows.datasets[0].raw.ch_names
eeg_indices = [ch_names.index(ch) for ch in eeg_ch]
ecg_indices = [ch_names.index(ch) for ch in ecg_ch]
print("EEG indices:", eeg_indices)
print("ECG indices:", ecg_indices)

# Separate EEG from ECG signals in one Dataset
train_paired = PairedEEGECGDataset(train_windows, eeg_indices, ecg_indices)
val_paired   = PairedEEGECGDataset(val_windows,   eeg_indices, ecg_indices)
test_paired  = PairedEEGECGDataset(test_windows,  eeg_indices, ecg_indices)
print('Paired Datasets created')

# DataLoaders
train_loader = DataLoader(train_paired, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_paired,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_paired,  batch_size=32, shuffle=False)
print('Data Loaders created!')

In [ ]:
# # Balance Classes (for classification phase)

# labels = [train_paired[i][2] for i in range(len(train_paired))]
# class_counts = np.bincount(labels)
# sample_weights = [1.0 / class_counts[l] for l in labels]
# sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
# print('Sampler created')

# train_loader_balanced = DataLoader(train_paired, batch_size=32, sampler=sampler)

Sampler was repeating minority class windows, the model was overfitting and causing a huge gap between train and validation. But since the DCCAE reconstrution loss does not depend on the labels we can pretrain the model with the class imbalance and then intoduce the sampler for classifier fine-tuning. 

In [ ]:
# CCA Loss taken and adapt from https://github.com/itsikad/cca-loss-TF/blob/master/src/cca_loss.py

def matrix_sqrt_inv(A):
    """
    Computes A^(-1/2) via eigendecomposition.
    """

    A = (A + A.T) / 2
    eigvals, eigvecs = torch.linalg.eigh(A)  
    

    eigvals = eigvals.clamp(min=1e-8)
    
    sqrt_inv = eigvecs @ torch.diag(1.0 / torch.sqrt(eigvals)) @ eigvecs.T
    return sqrt_inv


class CCA_Loss(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x1, x2, r1=1e-4, r2=1e-4):
        N = x1.shape[0]
        N_float = float(N)
        scale_factor = 1.0 / (N_float - 1.0)
        device = x1.device
        dtype = x1.dtype

        scale_mat = torch.eye(N, device=device, dtype=dtype) - (1.0 / N_float) * torch.ones((N, N), device=device, dtype=dtype)

        h1_bar = x1.T @ scale_mat
        h2_bar = x2.T @ scale_mat

        cov_11 = scale_factor * (h1_bar @ h1_bar.T) + r1 * torch.eye(x1.shape[1], device=device, dtype=dtype)
        cov_22 = scale_factor * (h2_bar @ h2_bar.T) + r2 * torch.eye(x2.shape[1], device=device, dtype=dtype)
        cov_12 = scale_factor * (h1_bar @ h2_bar.T)

        
        cov_11_sqrt_inv = matrix_sqrt_inv(cov_11)
        cov_22_sqrt_inv = matrix_sqrt_inv(cov_22)

        R = cov_11_sqrt_inv @ cov_12 @ cov_22_sqrt_inv

        U, s, Vh = torch.linalg.svd(R, full_matrices=False)
        V = Vh.T

        loss = -torch.sum(s)

        ctx.save_for_backward(
            h1_bar, h2_bar,
            cov_11_sqrt_inv, cov_22_sqrt_inv,
            U, s, V,
            torch.tensor(scale_factor, device=device, dtype=dtype)
        )

        return loss

    @staticmethod
    def backward(ctx, grad_output):
        h1_bar, h2_bar, cov_11_sqrt_inv, cov_22_sqrt_inv, U, s, V, scale_factor = ctx.saved_tensors

        cov_11_sqrt_inv_u = cov_11_sqrt_inv @ U
        cov_22_sqrt_inv_v = cov_22_sqrt_inv @ V

        S = torch.diag(s)

        delta_11 = -0.5 * cov_11_sqrt_inv_u @ S @ cov_11_sqrt_inv_u.T
        delta_22 = -0.5 * cov_22_sqrt_inv_v @ S @ cov_22_sqrt_inv_v.T
        delta_12 = cov_11_sqrt_inv @ U @ V.T @ cov_22_sqrt_inv

        grad_h1 = -scale_factor * (2.0 * delta_11 @ h1_bar + delta_12 @ h2_bar)
        grad_h2 = -scale_factor * (2.0 * delta_22 @ h2_bar + delta_12.T @ h1_bar)

        grad_x1 = grad_h1.T * grad_output
        grad_x2 = grad_h2.T * grad_output

        return grad_x1, grad_x2, None, None


def cca_loss(x1, x2, r1=1e-4, r2=1e-4):
    return CCA_Loss.apply(x1, x2, r1, r2)

## CNN-DCCAE model

First I replicated the model of the paper:

    - Encoder: 4 conv blocks with MaxPool, no activation (linear);
    - Decoder: transposed convolutions;
    - CCA loss;

The model was overfitting and not learning anything.

Fixes:

    - CCA loss was negative so I've changed the loss function to: total_loss = eeg_loss + ecg_loss - lambda_r * abs(correlation)

    - Increased lambda_r from 1e-10 (paper value) to 1e-3 to make CCA contribution more meaningful

    - Added LeakyRelu activation to prevent dead neurons while enabling nonlinear learning (Paper only uses linear activations bu it was causing near-identity transformations and the model stuck at loss ~1.83; Then I tried ReLU but the ECG decoder was collapsing to zero)

    - added a different and simpler decoder for the ECG for a single-channel signal reconstruction. (the original decoder was shared with EEG and ECG and the ECG decoder was collapsing)

    - Added BatchNorm after each conv block to stabilise training

    - Added dropout for regularization so it could prevent overfitting.

    - For the optimizer I started using the Adam with the same learning rate and a global weight decay, as in the paper, but it was collapsing the EEG encoder weights to zero. So I'm only adding weight decay to the decoder to improve the reconstruction, but not on the encoder, in order to preserve the learned representations.

    - I'm starting with a lower learning rate, trying to prevent that the model gets stuck in a local minimum (or that the model overshoot optimal point in epoch 1) and added the schedule to change the learing rate while training and also early stopping

    - Increased the number of filters to 32/64 filters (Paper uses 10 filters which was stucking the loss at ~1.87 and it was predicting the signal mean), with this change the loss dropped to ~1.66.




In [ ]:
# Set the number of channels
n_channels = 19
num_samples = 5 * 256 # 5 sec with a fs of 256 Hz
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CNNEncoder(nn.Module):
    def __init__(self, n_channels, T=num_samples, dropout=0.3):
        super().__init__()
        self.encoder_block = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),  
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),          
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(5),
            nn.BatchNorm1d(64, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(64, 32, kernel_size=5, padding=2),          
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(5),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(32, 1, kernel_size=1),
            nn.Flatten(),
            nn.BatchNorm1d(T // (2 * 5 * 5), momentum=0.01)
        )
        # latent dim = T // (2*5*5) = 25
    def forward(self, x):
        return self.encoder_block(x)
        


class CNNDecoder(nn.Module):
    def __init__(self, out_channels, T=num_samples, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(32, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(64, 32, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(32, 16, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(16, 8, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(8, 8, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(8, 10, kernel_size=1),
            nn.Upsample(size=T),
            nn.Conv1d(10, out_channels, kernel_size=1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)   
        return self.convs(x)
    
class ECGDecoder(nn.Module):
    def __init__(self, T=num_samples, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            nn.ConvTranspose1d(1, 32, kernel_size=5, stride=2, padding=2, output_padding=1),  
            nn.BatchNorm1d(32, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(32, 64, kernel_size=5, stride=5, padding=2, output_padding=4), 
            nn.BatchNorm1d(64, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=5, padding=2, output_padding=4),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Upsample(size=T),                  
            nn.Conv1d(32, 1, kernel_size=1)         
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        return self.convs(x)

In [ ]:
class DCCAE(nn.Module):

    def __init__(self, n_eeg_channels = 19, T = num_samples, lambda_r = 1e-10):
        super().__init__()
        self.lambda_r = lambda_r

        self.eeg_encoder = CNNEncoder(n_channels=n_eeg_channels, T=T)
        self.ecg_encoder = CNNEncoder(n_channels=1, T=T)
        self.eeg_decoder = CNNDecoder(out_channels=n_eeg_channels, T=T)
        self.ecg_decoder = ECGDecoder(T=T)

    def encode(self, eeg, ecg):
        return self.eeg_encoder(eeg), self.ecg_encoder(ecg)
    
    def forward(self, eeg, ecg):
        z_eeg = self.eeg_encoder(eeg)
        z_ecg = self.ecg_encoder(ecg)
        r_eeg = self.eeg_decoder(z_eeg)
        r_ecg = self.ecg_decoder(z_ecg)

        return z_eeg, z_ecg, r_eeg, r_ecg
    
    # Loss function
    def loss(self, eeg, ecg, lambda_r = 1e-3):
        """
        lambda_r = 1e-10 to tune the model and then increased to 1e-3 so that the correlation has more weight
        DCCAE loss = MSE(EEG) + MSE(ECG) - abs(lambda_r) * CCA_loss
        """
        z_eeg, z_ecg, r_eeg, r_ecg = self.forward(eeg, ecg)

        # Reconstruction losses using Mean Squared Error 
        eeg_loss = F.mse_loss(r_eeg, eeg)
        ecg_loss = F.mse_loss(r_ecg, ecg)

        # Correlation loss
        correlation = cca_loss(z_eeg, z_ecg)
        #correlation = cossine_similarity_loss(z_eeg, z_ecg)    # did not try this correlation metric

        # Loss function (I'm using the absolute value of the correlation because I checked that the cca loss is negative and since we want to maximize it it needed to be positive in this formula)
        total_loss = eeg_loss + ecg_loss - lambda_r * abs(correlation)

        return total_loss, {
            "total_loss": total_loss.item(),
            "eeg_loss": eeg_loss.item(),
            "ecg_loss": ecg_loss.item(),
            "correlation_loss": correlation.item(),
        }

    
model = DCCAE().to(device)
print(model)


In [ ]:
# Sanity check
model.train()
eeg, ecg, y, _ = next(iter(train_loader))
eeg, ecg = eeg.to(device), ecg.to(device)

print("EEG input shape:", eeg.shape)   
print("ECG input shape:", ecg.shape)   

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
optimizer.zero_grad()
loss, loss_dict = model.loss(eeg, ecg)

print("Loss:", loss_dict)
print("Total loss:", loss.item())

loss.backward()

total_grad = sum(p.grad.abs().sum().item() for p in model.parameters() if p.grad is not None)
print("Total gradient magnitude:", total_grad)

# Check model and reconstructions 
with torch.no_grad():
    z_eeg, z_ecg, r_eeg, r_ecg = model(eeg, ecg)
    print("z_eeg shape:", z_eeg.shape)  
    print("z_ecg shape:", z_ecg.shape)  
    print("r_eeg shape:", r_eeg.shape)  
    print("r_ecg shape:", r_ecg.shape)   
    print("Input EEG mean/std:", eeg.mean().item(), eeg.std().item())
    print("Recon EEG mean/std:", r_eeg.mean().item(), r_eeg.std().item())
    print("Input ECG mean/std:", ecg.mean().item(), ecg.std().item())
    print("Recon ECG mean/std:", r_ecg.mean().item(), r_ecg.std().item())

## Unsupervised training

In [ ]:
model = DCCAE().to(device)

optimizer = torch.optim.Adam([
    {'params': model.eeg_encoder.parameters(), 'weight_decay': 0.0},
    {'params': model.ecg_encoder.parameters(), 'weight_decay': 0.0}, # No weight decay on the encoders
    {'params': model.eeg_decoder.parameters(), 'weight_decay': 1e-4}, # Added only on the decoders
    {'params': model.ecg_decoder.parameters(), 'weight_decay': 1e-4},
], lr=3e-4) # start from reduced lr, before was lr=1e-3

# Scheduler to lower the learning rate if the model do not improve
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

epochs = 30
patience = 10
epochs_no_improve = 0

In [ ]:
# train_losses = []
# val_losses = []
# best_val_loss = float('inf')
# epochs_no_improve = 0


# for epoch in range(epochs):
#     model.train()
#     total_train_loss = 0.0

#     for eeg, ecg, y, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
#         eeg = eeg.to(device)
#         ecg = ecg.to(device)

#         optimizer.zero_grad()
#         loss, loss_dict = model.loss(eeg, ecg)

#         loss.backward()
#         # prevent gradients from becoming too large and explode
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#         total_train_loss += loss.item()

#     avg_train_loss = total_train_loss / len(train_loader)
#     train_losses.append(avg_train_loss)
    

#     # Validation
#     model.eval()
#     total_val_loss = 0.0

#     with torch.no_grad():
#         for eeg, ecg, y, _ in val_loader:
#             eeg = eeg.to(device)
#             ecg = ecg.to(device)
#             loss, _ = model.loss(eeg, ecg)
#             total_val_loss += loss.item()

#     avg_val_loss = total_val_loss / len(val_loader)
    
    
#     val_losses.append(avg_val_loss)

#     scheduler.step(avg_val_loss)

#     # Print current lr
#     current_lr = optimizer.param_groups[0]['lr']
#     print(f'Epoch {epoch+1} - Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {current_lr:.2e}')
    

#     if avg_val_loss < best_val_loss:
#         best_val_loss = avg_val_loss
#         torch.save(model.state_dict(), 'best_dccae_lr.pth')
#         print(f'New best model saved (val loss: {best_val_loss:.4f})')
#         epochs_no_improve = 0
    
#     else:
#         epochs_no_improve += 1
#         if epochs_no_improve >= patience:
#             print(f'Early stopping at epoch {epoch+1}')
#             break

# plt.figure(figsize=(8, 5))
# plt.plot(train_losses, label='Train Loss')
# plt.plot(val_losses, label='Val Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Curve')
# plt.legend()
# plt.show()

## Evaluation of the CNN-DCCAE model

In [ ]:
from collections import defaultdict

# Load the model
model.load_state_dict(torch.load('best_dccae_lr.pth', weights_only=True))
model.eval()
metrics = defaultdict(list)


# Evaluate on test set
with torch.no_grad():
    for eeg, ecg, y, _ in tqdm(test_loader, desc='test set'):
        eeg, ecg = eeg.to(device), ecg.to(device)
        _, loss_dict = model.loss(eeg, ecg)

        for k, v in loss_dict.items():
            metrics[k].append(float(v))

for k, vals in metrics.items():
    print(
        f"{k}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}"
    )

# Encode test and train sets

The encoded test set will be used to evaluate the clusters on the latent space

The encoded test set + train set will be used for the downstream classifiers

A subject-wise evaluation is also performed through majority voting

In [ ]:
# Encode test set
model.eval()
all_z_eeg, all_z_ecg, all_y = [], [], []

with torch.no_grad():
    for eeg, ecg, y, _ in tqdm(test_loader, desc='Encoding test set'):
        eeg, ecg = eeg.to(device), ecg.to(device)
        z_eeg, z_ecg = model.encode(eeg, ecg)
        all_z_eeg.append(z_eeg.cpu())
        all_z_ecg.append(z_ecg.cpu())
        all_y.append(y)

# save the independent representations
all_z_eeg = torch.cat(all_z_eeg).numpy()
all_z_ecg = torch.cat(all_z_ecg).numpy()

# true labels
all_y = torch.cat(all_y).numpy()

# Concat EEG and ECG representations
z_fused = np.concatenate([all_z_eeg, all_z_ecg], axis=1)

In [ ]:
# Encode train set too
all_z_eeg_train, all_z_ecg_train, all_y_train = [], [], []
plot_loader_train = DataLoader(train_paired, batch_size=32, shuffle=False)

model.eval()
with torch.no_grad():
    for eeg, ecg, y, _ in tqdm(plot_loader_train, desc='Encoding train set'):
        eeg, ecg = eeg.to(device), ecg.to(device)
        z_eeg, z_ecg = model.encode(eeg, ecg)
        all_z_eeg_train.append(z_eeg.cpu())
        all_z_ecg_train.append(z_ecg.cpu())
        all_y_train.append(y)

# save the independent representations
all_z_eeg_train = torch.cat(all_z_eeg_train).numpy()
all_z_ecg_train = torch.cat(all_z_ecg_train).numpy()

# true labels
all_y_train     = torch.cat(all_y_train).numpy()

# Concat EEG and ECG representations
z_fused_train   = np.concatenate([all_z_eeg_train, all_z_ecg_train], axis=1)

In [ ]:
# Get subject ID for each window in test set
test_subjects = []
for ds in test_windows.datasets:
    subj = get_subject_id(ds)
    n_windows = len(ds)
    test_subjects.extend([subj] * n_windows)

test_subjects = np.array(test_subjects)
print(f'Total test windows: {len(test_subjects)}')
print(f'Unique test subjects: {len(set(test_subjects))}')


print(f'all_y length: {len(all_y)}')


## Embedding quality metrics

We are gonna evaluate the resulting embeddings trough silhouette, davies_bouldin and calinski_harabasz metrics

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Compute on test set embeddings
for name, z in [('EEG', all_z_eeg), ('ECG', all_z_ecg), ('Fused', z_fused)]:
    sil = silhouette_score(z, all_y, sample_size=10000, random_state=42)
    db  = davies_bouldin_score(z, all_y)
    ch  = calinski_harabasz_score(z, all_y)
    print(f'{name}: Silhouette={sil:.4f} | Davies-Bouldin={db:.4f} | Calinski-Harabasz={ch:.1f}')

## Clustering Evaluation

For the cluster evaluation I'm testing first with KMeans for a kick evaluations (because its faster) and then I test it with SpectralClustering as in the paper (it takes a long time to run on large datasets so I've only performed it on a 10,000 balanced subsample).


The evaluation metrics are then the Clustering accuracy and the RIOC (to compare with the paper)

In [ ]:
from sklearn.cluster import SpectralClustering, MiniBatchKMeans
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.optimize import linear_sum_assignment

def clustering_accuracy(y_true, y_pred):
    """Match cluster labels to true labels using Hungarian algorithm."""
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm) # Hungarian matching to find the best mapping
    matched = np.zeros_like(y_pred)
    for r, c in zip(row_ind, col_ind):
        matched[y_pred == c] = r
    return accuracy_score(y_true, matched), matched


def evaluate_clustering(z, y, name, n_subsample=10000):
    """  
    Evaluate clustering performance using K-Means and Spectral Clustering (on a subsample).

    Returns the clustering accuracy and the RIOC metrics for each method
    """
    print(f'\n── {name} ──')

    # KMeans on full data (fast but less acurate)
    kmeans = MiniBatchKMeans(n_clusters=2, random_state=42)
    km_labels = kmeans.fit_predict(z)
    km_acc, km_matched = clustering_accuracy(y, km_labels)  
    km_rioc = (km_acc - 0.5) / 0.5 * 100
    print(f'KMeans — Acc: {km_acc*100:.3f}% | RIOC: {km_rioc:.2f}%')
    print(classification_report(y, km_matched, target_names=['Non-epileptic', 'Epileptic']))

    # Spectral on balanced subsample (matches paper methodology) (it takes longer to run so I'll only run it in a subsample)
    
    # Get the same number of epileptic and non epilepic subject for the subsample so it can be balanced
    idx_epi = np.where(y == 1)[0]
    idx_non = np.where(y == 0)[0]
    n_each  = min(n_subsample // 2, len(idx_non))
    idx_sub = np.concatenate([
        np.random.choice(idx_epi, n_each, replace=False),
        np.random.choice(idx_non, n_each, replace=False)
    ])
    np.random.shuffle(idx_sub)

    # subsample
    z_sub = z[idx_sub]
    y_sub = y[idx_sub]

    # Spectral Clustering
    sc = SpectralClustering(n_clusters=2, affinity='nearest_neighbors',
                            n_neighbors=10, random_state=42, n_jobs=-1) # using 10 neighbors
    sc_labels = sc.fit_predict(z_sub)
    sc_acc, sc_matched = clustering_accuracy(y_sub, sc_labels)
    sc_rioc = (sc_acc - 0.5) / 0.5 * 100
    print(f'Spectral — Acc: {sc_acc*100:.3f}% | RIOC: {sc_rioc:.2f}% (n={len(idx_sub)})')

    return km_acc, km_rioc, sc_acc, sc_rioc, km_matched, idx_sub, sc_matched


# Run for all three embeddings

embeddings_test = {
    'EEG':   all_z_eeg,
    'ECG':   all_z_ecg,
    'Fused': z_fused
}

# save the results
clustering_results = {}

for name, z in embeddings_test.items():
    km_acc, km_rioc, sc_acc, sc_rioc, km_matched, idx_sub, sc_matched = evaluate_clustering(z, all_y, name)
    clustering_results[name] = {
        'km_matched': km_matched,   # window-level, full test set
        'idx_sub':    idx_sub,      # indices used for spectral
        'sc_matched': sc_matched    # window-level, subsample only
    }

print('\nPaper baseline (Fused, Spectral): Acc=68.704% | RIOC=37.41%')

## Subject-level evaluation

The subject-level evaluation is performed by majority voting of the windows. Meaning that, predictions from multiple samples belonging to the same subject are aggregated using majority voting to obtain a single prediction per subject. Subject-level metrics (Balanced accuracy and macro F1) and a confusion matrix are computed.


In [ ]:
# Subject-level for clustering

from sklearn.metrics import (balanced_accuracy_score, f1_score, 
                             confusion_matrix, classification_report)
import seaborn as sns

def subject_level_eval(y_pred, y_true, subjects, name=''):
    """ 
    Evaluate classification performance at the subject level by majority voting.

    Returns confusion matrix, balanced accuracy, weighted macro, macro F1, classification report and subject's true and predicted label
    
    """
    subject_preds = {}
    subject_true  = {}
    
    for subj, pred, true in zip(subjects, y_pred, y_true):
        # Subject appear for the first time
        if subj not in subject_preds:
            subject_preds[subj] = []
            subject_true[subj]  = int(true)
        # Add pediction to the dict
        subject_preds[subj].append(int(pred))
    
    subj_ids = list(subject_preds.keys())

    # Majority vote
    subj_pred_final = [Counter(subject_preds[s]).most_common(1)[0][0] for s in subj_ids]  # finds the most frequent prediction for each subject.
    subj_true_final = [subject_true[s] for s in subj_ids]  # True labels
    
    # Metrics
    bal_acc = balanced_accuracy_score(subj_true_final, subj_pred_final)
    f1_mac  = f1_score(subj_true_final, subj_pred_final, average='macro')
    f1_w    = f1_score(subj_true_final, subj_pred_final, average='weighted')
    cm      = confusion_matrix(subj_true_final, subj_pred_final)
    
    print(f'\n── Subject-level: {name} ──')
    print(f'Subjects: {len(subj_ids)} | Bal. Acc: {bal_acc*100:.2f}% | Macro F1: {f1_mac:.4f}')
    print(classification_report(subj_true_final, subj_pred_final,
                                target_names=['Non-epileptic', 'Epileptic']))
    
    # Confusion matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['Non-epileptic', 'Epileptic'],
                yticklabels=['Non-epileptic', 'Epileptic'],
                cmap='Blues')
    plt.title(f'Subject-level CM — {name}')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    
    return bal_acc, f1_mac, f1_w, subj_true_final, subj_pred_final


# Save the results

subj_results = {}

for key, res in clustering_results.items():
    bal_acc, f1_mac, f1_w, y_true_s, y_pred_s = subject_level_eval(
        res['km_matched'], all_y, test_subjects, name=f'KMeans {key}'
    )
    subj_results[key] = {
        'bal_acc': bal_acc,
        'f1_mac':  f1_mac,
        'f1_w':    f1_w
    }

# Subject-level for Spectral (subsample only so subjects may be incomplete)
# Note: spectral was on a subsample so subject coverage is partial
for name, res in clustering_results.items():
    subject_level_eval(
        res['sc_matched'], 
        all_y[res['idx_sub']], 
        test_subjects[res['idx_sub']], 
        name=f'Spectral {name} (subsample)'
    )

## Downstream classifiers

Furthemore, we use downstream classifiers (RandomForestClassifier, LogisticRegression and LinearSVC) to further evaluate the discriminative content of the embeddings

In [ ]:
from sklearn.utils import resample
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV

# Subsample to 10k balanced for slow classifiers
def get_balanced_subsample(z, y, n_per_class=5000):
    """ Creates a balanced subsample of the dataset """
    idx_epi  = np.where(y == 1)[0]
    idx_non  = np.where(y == 0)[0]
    idx_epi_s  = resample(idx_epi,  n_samples=n_per_class, random_state=42)
    idx_non_s  = resample(idx_non,  n_samples=n_per_class, random_state=42)
    idx = np.concatenate([idx_epi_s, idx_non_s])
    return z[idx], y[idx]

# Define classifiers — LinearSVC instead of SVC (much faster)
classifiers = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'LinearSVM':           LinearSVC(class_weight='balanced', max_iter=2000),
    'Random Forest':       RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1)
}

In [ ]:
# Evaluate on each embedding
embeddings = {
    'EEG':   (all_z_eeg_train,   all_z_eeg),
    'ECG':   (all_z_ecg_train,   all_z_ecg),
    'Fused': (z_fused_train,     z_fused)
}


results = {}

for emb_name, (z_train, z_test) in embeddings.items():
    # Standardize embeddings
    scaler = StandardScaler()
    z_train_sc = scaler.fit_transform(z_train)
    z_test_sc  = scaler.transform(z_test)

    # Subsample train for slow classifiers
    z_train_sub, y_train_sub = get_balanced_subsample(z_train_sc, all_y_train)

    for clf_name, clf in classifiers.items():
        print(f'\nFitting {emb_name} + {clf_name}...')

        # Use subsampled data for SVM and RF
        if clf_name in ['LinearSVM', 'Random Forest']:
            clf.fit(z_train_sub, y_train_sub)
        else:
            clf.fit(z_train_sc, all_y_train)

        # Generate predictions on the test set
        y_pred = clf.predict(z_test_sc)

        # Obtain prediction probabilities for AUROC computation
        # LinearSVC requires probability calibration
        if clf_name == 'LinearSVM':
            cal_clf = CalibratedClassifierCV(clf, cv='prefit')
            cal_clf.fit(z_train_sub, y_train_sub)
            y_prob = cal_clf.predict_proba(z_test_sc)[:, 1]
        else:
            y_prob = clf.predict_proba(z_test_sc)[:, 1]

        # Compute evaluation metrics and store results
        bal_acc = balanced_accuracy_score(all_y, y_pred)
        auroc   = roc_auc_score(all_y, y_prob)

        results[f'{emb_name}_{clf_name}'] = {
            'bal_acc': bal_acc,
            'auroc':   auroc,
            'y_pred':  y_pred,
            'y_prob':  y_prob
        }

        print(f'  Balanced Acc: {bal_acc:.4f} | AUROC: {auroc:.4f}')
        print(classification_report(all_y, y_pred,
              target_names=['Non-epileptic', 'Epileptic']))

### Subject-level evaluation for downstream classifiers

In [ ]:
# Subject-level for classifiers

subj_results_clf = {}

for key, res in results.items():
    bal_acc, f1_mac, f1_w, y_true_s, y_pred_s = subject_level_eval(
        res['y_pred'], all_y, test_subjects, name=key
    )
    subj_results_clf[key] = {
        'bal_acc': bal_acc,
        'f1_mac':  f1_mac,
        'f1_w':    f1_w
    }

# Cluster visualization

We plotted the representations of each embedding using dimensional reduction techniques (UMAP, t-SNE and PCA)

First we'll check how the KMeans classifier tried to separate the two classes using UMAP

In [ ]:
import umap

reducer_fused = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
z_fused_2d = reducer_fused.fit_transform(z_fused)

cluster_labels = clustering_results['Fused']['km_matched']

df = pd.DataFrame({
    'x':       z_fused_2d[:, 0],
    'y':       z_fused_2d[:, 1],
    'label':   ['Epileptic' if y == 1 else 'Non-epileptic' for y in all_y],
    'cluster': [f'Cluster {c}' for c in cluster_labels],
    'subject': test_subjects
})

# True labels
fig1 = px.scatter(df, x='x', y='y', color='label',
                  hover_data=['subject'],
                  color_discrete_map={'Epileptic': 'tomato', 'Non-epileptic': 'steelblue'},
                  opacity=0.5, title='UMAP — True Labels (Test Set)')
fig1.update_traces(marker=dict(size=3))
fig1.show()

# Cluster assignments
fig2 = px.scatter(df, x='x', y='y', color='cluster',
                  hover_data=['subject', 'label'],
                  opacity=0.5, title='UMAP — KMeans Clusters (Test Set)')
fig2.update_traces(marker=dict(size=3))
fig2.show()

By this representaions we can see that the classes are very mixed, so the encoder has not learned.

By the spectral clustering, it has found something real, but not related to the separation between epileptic and non-epileptic.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def visualize_embeddings(z, labels, title):
    """
    Visualize high-dimensional embeddings using PCA, t-SNE, and UMAP.

    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Subsample consistently for all methods
    n_sub = 10000
    idx_epi = np.where(labels == 1)[0]
    idx_non = np.where(labels == 0)[0]
    n_each  = min(n_sub // 2, len(idx_non))  # balanced subsample
    idx_sub = np.concatenate([
        np.random.choice(idx_epi, n_each, replace=False),
        np.random.choice(idx_non, n_each, replace=False)
    ])
    
    z_sub      = z[idx_sub]
    labels_sub = labels[idx_sub]

    # PCA
    z_pca = PCA(n_components=2).fit_transform(z_sub)

    # t-SNE
    z_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(z_sub)

    # UMAP
    z_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(z_sub)

    for ax, z_2d, method in zip(axes,
                                 [z_pca, z_tsne, z_umap],
                                 ['PCA', 't-SNE', 'UMAP']):
        # Plot epileptic first, non-epileptic on top
        idx_e = labels_sub == 1
        idx_n = labels_sub == 0
        
        ax.scatter(z_2d[idx_e, 0], z_2d[idx_e, 1], 
                   c='tomato', alpha=0.3, s=2, label='Epileptic')
        ax.scatter(z_2d[idx_n, 0], z_2d[idx_n, 1], 
                   c='steelblue', alpha=0.6, s=4, label='Non-epileptic')
        
        ax.set_title(f'{title} — {method}')
        ax.set_xlabel('Component 1')
        ax.set_ylabel('Component 2')
        ax.legend(markerscale=4, fontsize=8)

    plt.tight_layout()
    plt.show()

# Plot all three embeddings
visualize_embeddings(all_z_eeg,  all_y, 'EEG Embeddings')
visualize_embeddings(all_z_ecg,  all_y, 'ECG Embeddings')
visualize_embeddings(z_fused,    all_y, 'Fused Embeddings')

In [ ]:
import pandas as pd

rows = []

# Clustering results
for name, res in clustering_results.items():
    # KMeans window-level
    km_acc, km_matched = clustering_accuracy(all_y, res['km_matched'])
    km_rioc = (km_acc - 0.5) / 0.5 * 100
    
    # KMeans subject-level (already computed in subj_results)
    km_subj = subj_results.get(name, {})
    
    rows.append({
        'Method':          'KMeans',
        'Embedding':       name,
        'Window Acc':      f'{km_acc*100:.2f}%',
        'Window RIOC':     f'{km_rioc:.2f}%',
        'Window Bal.Acc':  '—',
        'Window AUROC':    '—',
        'Window Macro F1': '—',
        'Subj Bal.Acc':    f"{km_subj.get('bal_acc', 0)*100:.2f}%",
        'Subj Macro F1':   f"{km_subj.get('f1_mac', 0):.4f}"
    })

# Downstream classifier results
for key, res in results.items():
    emb, clf = key.split('_', 1)
    subj = subj_results_clf.get(key, {})
    
    rows.append({
        'Method':          clf,
        'Embedding':       emb,
        'Window Acc':      '—',
        'Window RIOC':     '—',
        'Window Bal.Acc':  f"{res['bal_acc']*100:.2f}%",
        'Window AUROC':    f"{res['auroc']:.4f}",
        'Window Macro F1': '—',
        'Subj Bal.Acc':    f"{subj.get('bal_acc', 0)*100:.2f}%",
        'Subj Macro F1':   f"{subj.get('f1_mac', 0):.4f}"
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

# Save csv
df_summary.to_csv('results_summary.csv', index=False)


print('\nPaper baseline — CNN-DCCAE (Spectral, Fused): Acc=68.704% | RIOC=37.41%')

# Supervised fine-tuning

Supervised fine-tunning is used as a second training stage with the goal of guiding the encoders towards more discriminative embeddings

We are gonna freeze the encodersto preserve the learned representations and then we are adding a simple classifier (MLP) head

To perform this supervised training we now need to balance the training set.

In [ ]:
# Balance classes
labels = [train_paired[i][2] for i in range(len(train_paired))]
class_counts = np.bincount(labels)
sample_weights = [1.0 / class_counts[l] for l in labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader_balanced = DataLoader(train_paired, batch_size=32, sampler=sampler)

## Model

In [ ]:
class DCCAEClassifier(nn.Module):
    def __init__(self, dccae_model, latent_dim=25):
        super().__init__()
        self.eeg_encoder = dccae_model.eeg_encoder
        self.ecg_encoder = dccae_model.ecg_encoder

        # Freeze encoders
        for param in self.eeg_encoder.parameters():
            param.requires_grad = False
        for param in self.ecg_encoder.parameters():
            param.requires_grad = False

        # MLP head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim * 2, 64),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, eeg, ecg):
        with torch.no_grad():
            z_eeg = self.eeg_encoder(eeg)
            z_ecg = self.ecg_encoder(ecg)
        z_fused = torch.cat([z_eeg, z_ecg], dim=-1)
        return self.classifier(z_fused)
    

model.load_state_dict(torch.load('best_dccae_lr.pth', weights_only=True))
classifier = DCCAEClassifier(model).to(device)

### Supervised training

In [ ]:
# 9:1 imbalance
pos_weight = torch.tensor([9.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier.parameters()),
    lr=1e-3
)

epochs = 20
patience = 8
epochs_no_improve = 0

In [ ]:
# from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score

# train_losses = []
# val_losses = []
# best_bal_acc = 0.0
# epochs_no_improve = 0


# for epoch in range(epochs):
#     classifier.train()
#     total_train_loss = 0.0

#     for eeg, ecg, y, _ in tqdm(train_loader_balanced, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
#         eeg = eeg.to(device)
#         ecg = ecg.to(device)
#         y = y.float().to(device)

#         optimizer.zero_grad()
#         logits = classifier(eeg, ecg).squeeze(1) 

#         loss = criterion(logits, y)

#         loss.backward()

#         optimizer.step()
#         total_train_loss += loss.item()

#     avg_train_loss = total_train_loss / len(train_loader_balanced)
#     train_losses.append(avg_train_loss)
    

#     # Validation
#     classifier.eval()
#     total_val_loss = 0.0
    
#     all_logits, all_y_val = [], []
#     with torch.no_grad():
#         for eeg, ecg, y, _ in val_loader:
#             eeg, ecg = eeg.to(device), ecg.to(device)
#             y = y.float().to(device)
#             logits = classifier(eeg, ecg).squeeze(1)
#             loss = criterion(logits, y)
#             total_val_loss += loss.item()
#             all_logits.append(logits.cpu())
#             all_y_val.append(y.cpu())

#     avg_val_loss = total_val_loss / len(val_loader)
#     val_losses.append(avg_val_loss)

#     all_logits = torch.cat(all_logits)
#     all_y_val = torch.cat(all_y_val)
#     probs = torch.sigmoid(all_logits).numpy()
#     threshold = class_counts[0] / (class_counts[0] + class_counts[1])  # ~0.1 for 9:1 imbalance
#     preds = (probs > threshold).astype(int)

#     bal_acc = balanced_accuracy_score(all_y_val.numpy(), preds)
#     auroc = roc_auc_score(all_y_val.numpy(), probs)
#     print(f'Epoch {epoch+1} - Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | BalAcc: {bal_acc:.4f} | AUROC: {auroc:.4f}')


#     if bal_acc > best_bal_acc:
#         best_bal_acc = bal_acc
#         torch.save(classifier.state_dict(), 'best_dccae_classifier.pth')
#         print(f'New best model saved (bal acc: {best_bal_acc:.4f})')
#         epochs_no_improve = 0
    
#     else:
#         epochs_no_improve += 1
#         if epochs_no_improve >= patience:
#             print(f'Early stopping at epoch {epoch+1}')
#             break

# plt.figure(figsize=(8, 5))
# plt.plot(train_losses, label='Train Loss')
# plt.plot(val_losses, label='Val Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Curve')
# plt.legend()
# plt.show()  

## Find best threshold

We are going to find the threshold value, on the validation set, that maximizes the balanced accuracy

In [ ]:
from sklearn.metrics import roc_curve

classifier.load_state_dict(torch.load('best_dccae_classifier.pth', weights_only=True))
classifier.eval()

all_logits, all_y_val = [], []
with torch.no_grad():
    for eeg, ecg, y, _ in val_loader:
        eeg, ecg = eeg.to(device), ecg.to(device)
        logits = classifier(eeg, ecg).squeeze(1)
        all_logits.append(logits.cpu())
        all_y_val.append(y.cpu())

probs = torch.sigmoid(torch.cat(all_logits)).numpy()
y_val = torch.cat(all_y_val).numpy()

# Find threshold that maximises balanced accuracy
fpr, tpr, thresholds = roc_curve(y_val, probs)
balanced_accs = (tpr + (1 - fpr)) / 2
best_idx = np.argmax(balanced_accs)
best_threshold = thresholds[best_idx]
print(f'Best threshold: {best_threshold:.4f}')
print(f'Best balanced accuracy: {balanced_accs[best_idx]:.4f}')

## Evaluation with tuned threshold

In [ ]:
classifier.load_state_dict(torch.load('best_dccae_classifier.pth', weights_only=True))
classifier.eval()

all_logits, all_y_test = [], []
with torch.no_grad():
    for eeg, ecg, y, _ in test_loader:
        eeg, ecg = eeg.to(device), ecg.to(device)
        logits = classifier(eeg, ecg).squeeze(1)
        all_logits.append(logits.cpu())
        all_y_test.append(y.cpu())

y_prob_supervised = torch.sigmoid(torch.cat(all_logits)).numpy()


y_pred_supervised = (y_prob_supervised > best_threshold).astype(int)
bal_acc_tuned = balanced_accuracy_score(all_y, y_pred_supervised)
print(f'Balanced accuracy with tuned threshold: {bal_acc_tuned:.4f}')

In [ ]:
# Metrics

bal_acc = balanced_accuracy_score(all_y, y_pred_supervised)
auroc   = roc_auc_score(all_y, y_prob_supervised)
print(f'Balanced accuracy: {bal_acc:.4f}')
print(f'AUROC: {auroc:.4f}')
print(classification_report(all_y, y_pred_supervised,
      target_names=['Non-epileptic', 'Epileptic']))


In [ ]:
# Subject-level evaluation
bal_acc, f1_mac, f1_w, _, _ = subject_level_eval(
    y_pred_supervised, all_y, test_subjects, name='MLP Classifier'
)

# Summary ROC curves

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

for key, res in results.items():
    fpr, tpr, _ = roc_curve(all_y, res['y_prob'])  # now works
    plt.plot(fpr, tpr, label=f"{key} (AUC={res['auroc']:.3f})", alpha=0.7)

fpr, tpr, _ = roc_curve(all_y, y_prob_supervised)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'MLP supervised (AUC={roc_auc:.3f})', linewidth=2, color='black')

plt.plot([0, 1], [0, 1], 'k--', label='Random chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Methods')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()